# Table 5 – Structured World Evaluation

This notebook compares **Hybrid Standard** and **ODRM\*** planners on three structured worlds.

For each world and agent count (50 and 100), we report:
- **Success Rate (%)**
- **Agent Completion Rate (%)**

Separate comparison tables are generated for **50 agents**, **100 agents**, and **per-world results**.

In [3]:
import re
import pandas as pd

In [4]:
hybrid_file = "Results_table_5/Hybrid_standard_structured_world_evaluation.txt"
odrm_file = "Results_table_5/odrmstar_structured_worlds_evaluation.txt"

In [5]:
def parse_hybrid(file_path):

    worlds = {}
    current_world = None

    with open(file_path, "r") as f:
        for line in f:

            if line.startswith("Structured_world"):
                current_world = line.strip()
                worlds[current_world] = {}

            if line.startswith("[EPISODE"):

                agent_match = re.search(r"Agents Completed:\s*(\d+)/(\d+)", line)
                success_match = re.search(r"Success:\s*(True|False)", line)

                if agent_match:

                    completed = int(agent_match.group(1))
                    total_agents = int(agent_match.group(2))

                    completion = (completed / total_agents) * 100

                    success = 100 if success_match.group(1) == "True" else 0

                    worlds[current_world][total_agents] = {
                        "success": success,
                        "completion": completion
                    }

    rows = []

    for world in worlds:
        for agents in worlds[world]:
            rows.append({
                "World": world,
                "Agents": agents,
                "Hybrid Success (%)": worlds[world][agents]["success"],
                "Hybrid Completion (%)": worlds[world][agents]["completion"]
            })

    return pd.DataFrame(rows)

In [6]:
def parse_odrm(file_path):

    worlds = {}
    current_world = None
    current_agents = None

    with open(file_path, "r") as f:
        for line in f:

            if line.startswith("Structured_world"):
                current_world = line.strip()
                worlds[current_world] = {}

            if "Agents:" in line:
                match = re.search(r"Agents:\s*(\d+)", line)
                if match:
                    current_agents = int(match.group(1))

            if line.startswith("[EPISODE"):

                if "std::bad_alloc" in line:
                    success = "OOM"
                else:
                    success = 100 if "Success: True" in line else 0

                worlds[current_world][current_agents] = success

    rows = []

    for world in worlds:
        for agents in worlds[world]:
            rows.append({
                "World": world,
                "Agents": agents,
                "ODRM* Success (%)": worlds[world][agents]
            })

    return pd.DataFrame(rows)

In [7]:
hybrid_df = parse_hybrid(hybrid_file)
odrm_df = parse_odrm(odrm_file)

In [8]:
comparison = hybrid_df.merge(odrm_df, on=["World", "Agents"])

comparison

,World,Agents,Hybrid Success (%),Hybrid Completion (%),ODRM* Success (%)
0,Structured_world_1,50,0,86.0,100
1,Structured_world_1,100,0,76.0,100
2,Structured_world_2,50,0,76.0,100
3,Structured_world_2,100,0,68.0,OOM
4,Structured_world_3,50,0,98.0,100
5,Structured_world_3,100,100,100.0,100


In [9]:
table_50 = comparison[comparison["Agents"] == 50]

table_50

,World,Agents,Hybrid Success (%),Hybrid Completion (%),ODRM* Success (%)
0,Structured_world_1,50,0,86.0,100
2,Structured_world_2,50,0,76.0,100
4,Structured_world_3,50,0,98.0,100


In [10]:
table_100 = comparison[comparison["Agents"] == 100]

table_100

,World,Agents,Hybrid Success (%),Hybrid Completion (%),ODRM* Success (%)
1,Structured_world_1,100,0,76.0,100
3,Structured_world_2,100,0,68.0,OOM
5,Structured_world_3,100,100,100.0,100
